In [ ]:
# =========================
# nb_anomaly_detection
# =========================
# Phase 9, SC-14: Detect anomalies in holdings (portfolio shifts, value spikes, unusual patterns)
# Uses AI to identify and classify anomalies with severity levels

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"
anomaly_min_prior_periods = 2  # Require at least 2 prior periods for baseline

# ---------- Imports ----------
from pyspark.sql import functions as F
import sys
import json
from datetime import datetime, timedelta

# ---------- Pre-flight checks ----------
try:
    mssparkutils
except NameError:
    print("[ERROR] mssparkutils not available - not running in Fabric context")
    sys.exit(1)

try:
    if not period or not period.strip():
        raise ValueError("period parameter is empty")
    if not run_id or not run_id.strip():
        raise ValueError("run_id parameter is empty")
    if anomaly_min_prior_periods < 1:
        raise ValueError("anomaly_min_prior_periods must be >= 1")
    print(f"[nb_anomaly_detection] START period={period}, run_id={run_id}")
except ValueError as e:
    print(f"[ERROR] Invalid parameters: {e}")
    sys.exit(1)

if not spark.catalog.tableExists("tpir_load_equivalent") or not spark.catalog.tableExists("ai_anomaly_events"):
    print("[ERROR] Required tables do not exist")
    sys.exit(1)

# Load shared_ai_utils
sys.path.append('/lakehouse/default/Files/config')
try:
    from shared_ai_utils import load_ai_config
    ai_enabled = True
except ImportError as e:
    print(f"[WARNING] shared_ai_utils not found: {e}")
    print("[nb_anomaly_detection] AI not available; skipping")
    mssparkutils.notebook.exit("AI_DISABLED")

# Load AI config
try:
    ai_client = load_ai_config('/lakehouse/default/Files/config/azure_openai_config.json')
    if ai_client is None:
        raise RuntimeError("AI config returned None")
except Exception as e:
    print(f"[ERROR] Failed to load AI config: {type(e).__name__}: {e}")
    mssparkutils.notebook.exit("CONFIG_ERROR")

# ---------- Load and validate data ----------
try:
    current_period = period
    prior_period_list = [
        (datetime.strptime(period, '%Y-%m') - timedelta(days=30*i)).strftime('%Y-%m') 
        for i in range(1, anomaly_min_prior_periods + 1)
    ]
    
    print(f"[nb_anomaly_detection] Current period: {current_period}")
    print(f"[nb_anomaly_detection] Prior periods: {', '.join(prior_period_list)}")
    
    current = spark.table("tpir_load_equivalent").filter(F.col("period") == current_period)
    if current.rdd.isEmpty():
        print(f"[WARNING] No data for current period={current_period}")
        mssparkutils.notebook.exit("NO_DATA")
    
    current_rows = current.count()
    print(f"[nb_anomaly_detection] Current period: {current_rows} rows")
    
    # Aggregate current policy totals
    current_agg = current.groupBy("dfm_id", "policy_id").agg(
        F.sum(F.col("market_value_gbp").cast("double")).alias("current_value")
    )
    
    # Get prior baseline (average of prior periods)
    prior_data = spark.table("tpir_load_equivalent").filter(
        F.col("period").isin(prior_period_list)
    )
    
    prior_rows = prior_data.count()
    print(f"[nb_anomaly_detection] Prior periods: {prior_rows} total rows")
    
    if prior_rows == 0:
        print("[WARNING] No prior period data available for baseline calculation")
        mssparkutils.notebook.exit("INSUFFICIENT_HISTORY")
    
    prior_agg = prior_data.groupBy("dfm_id", "policy_id").agg(
        F.avg(F.col("market_value_gbp").cast("double")).alias("baseline_value")
    )
    
    # Join and calculate deviation
    comparison = current_agg.join(prior_agg, ["dfm_id", "policy_id"], "left").fillna(0.0)
    with_deviation = comparison.withColumn(
        "pct_deviation",
        F.when(F.col("baseline_value") != 0,
               ((F.col("current_value") - F.col("baseline_value")) / F.col("baseline_value")) * 100)
        .otherwise(0.0)
    )
    
    # Flag extreme deviations
    anomalies = with_deviation.filter(
        (F.abs(F.col("pct_deviation")) > 40) |  # ±40% deviation
        (F.col("baseline_value") == 0)  # New holding
    )
    
    anomaly_count = anomalies.count()
    if anomaly_count == 0:
        print("[nb_anomaly_detection] No anomalies detected (within normal range)")
        mssparkutils.notebook.exit("OK")
    
    print(f"[nb_anomaly_detection] Detected {anomaly_count} potential anomalies...")

except Exception as e:
    print(f"[ERROR] Data loading failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)

# ---------- AI classification ----------
events = []
success_count = 0
error_count = 0

try:
    for row in anomalies.collect():
        dfm_id = str(row["dfm_id"]) if row["dfm_id"] else "UNKNOWN"
        policy_id = str(row["policy_id"]) if row["policy_id"] else "UNKNOWN"
        baseline = float(row["baseline_value"] or 0)
        current_val = float(row["current_value"] or 0)
        pct_dev = float(row["pct_deviation"] or 0)
        
        prompt = f"""
Analyze this portfolio anomaly:
- Policy: {policy_id} (DFM: {dfm_id})
- Baseline value (GBP): {baseline:.2f}
- Current value (GBP): {current_val:.2f}
- Change: {pct_dev:.1f}%

Classify as: portfolio_decrease, value_spike, unusual_pattern, or normal_variance
Rate severity: low, medium, high, or critical

Return JSON: {{"anomaly_type": "...", "severity": "...", "confidence": 0.85, "description": "..."}}
"""
        
        try:
            result_text = ai_client.detect_anomaly(prompt)
            
            if not result_text or not isinstance(result_text, str):
                raise ValueError("AI returned non-string response")
            
            result = json.loads(result_text)
            
            events.append({
                "period": period,
                "run_id": run_id,
                "dfm_id": dfm_id,
                "policy_id": policy_id,
                "security_id": "PORTFOLIO",
                "anomaly_type": str(result.get("anomaly_type", "unusual_pattern")),
                "severity": str(result.get("severity", "medium")),
                "baseline_value": baseline,
                "actual_value": current_val,
                "pct_deviation": pct_dev,
                "description": str(result.get("description", "")),
                "ai_confidence": float(result.get("confidence", 0.0)),
                "remediation_suggested": "review_holdings" if pct_dev < -30 else "check_market_data",
                "flagged_at": None
            })
            success_count += 1
        
        except json.JSONDecodeError as e:
            error_count += 1
            print(f"[WARNING] AI response not valid JSON for policy {policy_id}: {e}")
        except Exception as e:
            error_count += 1
            print(f"[WARNING] Error processing policy {policy_id}: {type(e).__name__}: {e}")

    # Write results
    if events:
        try:
            df_events = spark.createDataFrame(events)
            df_events.write.mode("append").saveAsTable("ai_anomaly_events")
            print(f"[nb_anomaly_detection] COMPLETE wrote {len(events)} anomaly events (success={success_count}, errors={error_count})")
        except Exception as e:
            print(f"[ERROR] Failed to write anomaly events: {type(e).__name__}: {e}")
            sys.exit(1)
    else:
        print(f"[WARNING] No valid anomalies recorded (processed {success_count}, errors {error_count})")
    
    mssparkutils.notebook.exit("OK")

except Exception as e:
    print(f"[ERROR] Anomaly detection failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)